# Day 5 — ILT 2: Partitioning Strategy & Storage Formats
### GlobalMart Data Engineering · 11:00 AM – 1:00 PM

---

## Session Objectives

By the end of this session you will be able to:
- Explain why storage format affects query performance
- Compare CSV, JSON, Parquet, and Delta — and choose the right one
- Define partitioning and explain how it enables partition pruning
- Choose the right partition column(s) for a Bronze table
- Explain the dangers of over-partitioning
- Describe Z-ORDER BY and when to use it
- Explain what Liquid Clustering is and how it differs from partitioning

---

## Agenda

| Time | Topic |
|------|-------|
| 11:00 | Storage formats — why format matters |
| 11:15 | CSV vs JSON vs Parquet vs Delta |
| 11:30 | Why Delta is the right choice for Bronze |
| 11:45 | What is partitioning? |
| 12:00 | Choosing partition columns |
| 12:15 | The over-partitioning trap |
| 12:25 | Z-ORDER BY |
| 12:40 | Liquid Clustering |
| 12:50 | Decision framework + Q&A |

---
## 1. Why Storage Format Matters

When Spark reads data, it scans files on ADLS. The format determines:
- **How much data is read** (column pruning, row skipping)
- **How fast it reads** (compression, encoding)
- **What guarantees you get** (ACID, schema enforcement)
- **How much storage you use** (compression ratio)

### The Cost of a Full Scan

```
Bronze orders table: 1.4 million rows, 9 columns

Query: SELECT COUNT(*) FROM bronze_orders WHERE OrderDate = '2026-06-15'

With CSV (no column pruning, no stats):
  → Reads ALL 9 columns for ALL 1.4M rows
  → 100% of data scanned

With Parquet (columnar, compressed):
  → Reads only OrderDate column
  → ~11% of data scanned

With Delta + partitioning (partition pruning):
  → Reads ONLY the partition for 2026-06-15
  → Maybe 0.1% of data scanned
```

> **Format choice is a multiplier on every query cost.**

---
## 2. Storage Format Comparison

| Feature | CSV | JSON | Parquet | Delta |
|---------|-----|------|---------|-------|
| **Type** | Row-based text | Row-based text | Columnar binary | Columnar binary + transaction log |
| **Compression** | None (or gzip) | None | Snappy / ZSTD | Snappy / ZSTD |
| **Schema enforcement** | No | Partial | Yes (in file) | Yes (enforced on write) |
| **Column pruning** | No — reads all columns | No | Yes | Yes |
| **Row skipping (stats)** | No | No | Partial | Yes (min/max per file) |
| **ACID transactions** | No | No | No | Yes |
| **Time travel** | No | No | No | Yes |
| **Schema evolution** | Manual | Manual | Limited | Yes (mergeSchema) |
| **Typical compression ratio** | 1x | 1x | 5–10x | 5–10x |
| **Typical read speed (vs CSV)** | 1x (baseline) | 0.8x | 3–8x | 3–8x |

---

### CSV — The Ingestion Format, Not the Storage Format

```
✅ CSV is good FOR: receiving files from external systems (suppliers, APIs)
❌ CSV is bad FOR: storing data in your Bronze/Silver/Gold layers

Problem with storing in CSV:
  - No schema enforcement → OrderID could be "ORDER-001" or "001" or 1
  - Every query reads every column
  - No statistics → Spark can't skip rows
  - File size bloats without compression
  - No transactions → partial writes leave corrupt data
```

### Parquet — The Columnar Standard

```
✅ Great for read-heavy analytics
✅ Column pruning → only read the columns you need
✅ Compressed → Snappy by default
❌ No ACID → two concurrent writes = corrupt data
❌ No time travel
❌ No easy schema evolution
❌ Updates require rewriting entire files
```

### Delta Lake — Parquet + Transaction Log

```
Delta = Parquet files + _delta_log/ folder

_delta_log/:
  00000000000000000000.json  ← Version 0: initial table creation
  00000000000000000001.json  ← Version 1: first write
  00000000000000000002.json  ← Version 2: OPTIMIZE run
  ...
  00000000000000000010.checkpoint.parquet  ← periodic checkpoint

Each log entry records:
  - Which files were added
  - Which files were removed
  - Min/max statistics per column per file
  - Schema at this version
  - Who made the change and when
```

**Delta = Parquet performance + ACID + time travel + schema evolution**

> **Use Delta for ALL Bronze, Silver, and Gold tables. No exceptions.**

---
## 3. Why Delta Is the Right Choice for Bronze

### 1. ACID Guarantees

```
Scenario: Auto Loader is mid-write to Bronze when the cluster dies.

Without ACID (Parquet/CSV):
  → Partial file written → 50,000 rows half-written → data corrupt
  → No way to know which rows made it

With Delta:
  → Transaction was not committed → Delta treats it as if it never happened
  → Next run restarts from checkpoint → clean, correct state
```

### 2. Time Travel

```sql
-- Read Bronze as it was yesterday (before a bad batch)
SELECT COUNT(*) FROM bronze_adls_orders
VERSION AS OF 5;

-- Or by timestamp:
SELECT * FROM bronze_adls_orders
TIMESTAMP AS OF '2026-06-10 12:00:00';
```

### 3. Schema Evolution

```python
# New column in source? Just add mergeSchema:
df.write.format("delta").mode("append").option("mergeSchema", "true").save(path)
# Bronze table automatically gains the new column
```

### 4. File Statistics for Query Optimisation

```
Delta stores min/max per column per file in the transaction log.

Query: WHERE OrderDate = '2026-06-15'

Delta checks transaction log:
  File A: OrderDate min=2026-06-01, max=2026-06-14 → SKIP
  File B: OrderDate min=2026-06-15, max=2026-06-20 → READ
  File C: OrderDate min=2026-06-21, max=2026-06-30 → SKIP

Result: only File B is read → massive I/O saving
```

This is called **data skipping** — and it's automatic with Delta.

---
## 4. What Is Partitioning?

Partitioning = physically splitting a table into **sub-folders** based on a column's value.

```
Without partitioning:
bronze/adls/orders/
  part-00000.parquet   ← all 1.4M rows in one folder
  part-00001.parquet
  part-00002.parquet

With partitioning by ingestion_date:
bronze/adls/orders/
  ingestion_date=2026-06-13/
    part-00000.parquet   ← only rows from June 13
  ingestion_date=2026-06-14/
    part-00000.parquet   ← only rows from June 14
  ingestion_date=2026-06-15/
    part-00000.parquet   ← only rows from June 15
```

### Partition Pruning

```
Query: SELECT * FROM bronze_adls_orders WHERE ingestion_date = '2026-06-15'

With partitioning:
  Spark looks at folder structure
  → Finds ingestion_date=2026-06-15/
  → Reads ONLY that folder
  → Skips ingestion_date=2026-06-13/ and ingestion_date=2026-06-14/

Result: if you have 30 days of data, Spark reads 1/30 = 3.3% of data
```

### How to Write a Partitioned Delta Table

```python
from pyspark.sql.functions import to_date, current_timestamp

df = df.withColumn("ingestion_date", to_date(current_timestamp()))

df.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("ingestion_date") \
    .save(bronze_path)
```

---
## 5. Choosing Partition Columns

### The 3 Criteria for a Good Partition Column

1. **High query filter frequency** — it's in your WHERE clause almost every time
2. **Low-to-medium cardinality** — not too many unique values
3. **Evenly distributed** — data spread roughly equally across partitions

---

### Good Partition Column Choices for Bronze

| Column | Cardinality | Good? | Reason |
|--------|------------|-------|--------|
| `ingestion_date` | ~365 values/year | ✅ Yes | Most queries filter by date; daily batch → natural partition |
| `OrderChannel` | 3-5 values | ✅ Yes | Low cardinality; if frequently filtered |
| `SupplierID` | 10-50 values | ✅ Maybe | Only if you often query per supplier |
| `CustomerID` | Millions | ❌ No | Too high — millions of tiny partitions |
| `OrderID` | Millions | ❌ No | Never partition on a primary key |
| `OrderDate` | Thousands | ❌ Careful | Use only if daily queries and data volume justifies it |

### GlobalMart Bronze Partitioning Strategy

```
bronze_supabase_orders:     partitionBy("ingestion_date")
bronze_api_promotions:      partitionBy("ingestion_date")
bronze_neo4j_supplier:      partitionBy("ingestion_date")  ← daily snapshot
bronze_adls_orders:         partitionBy("ingestion_date")
```

> **For Bronze, `ingestion_date` is almost always the right partition column.**
> Queries against Bronze typically ask: "show me what was ingested on date X".

---
## 6. The Over-Partitioning Trap

Partitioning creates one folder per unique value. Too many folders = **the small file problem**.

### What Happens When You Over-Partition

```
Scenario: Partition bronze_adls_orders by OrderID
  1,400,000 rows → 1,400,000 partitions → 1,400,000 folders
  Each folder: 1 file with 1 row

Problems:
  1. Metadata overhead: Delta transaction log tracks 1.4M files → slow to open table
  2. Spark task overhead: 1 Spark task per partition = 1.4M tasks for a simple COUNT
  3. ADLS list calls: listing 1.4M folders takes minutes
  4. Small files: ADLS is optimised for large files (128MB+), not tiny 1-row files
```

### The Small File Problem

```
Ideal file size: 128 MB – 1 GB (for Parquet/Delta on ADLS)

Small file scenario:
  1000 files × 100 KB = 100 MB total data
  vs
  1 file × 100 MB = 100 MB total data

Reading 1000 small files:
  → 1000 ADLS open() calls
  → 1000 Spark tasks
  → Overhead dominates actual data processing time
  → Queries are slow despite small data volume

Reading 1 large file:
  → 1 ADLS open() call
  → 1 Spark task (or few with parallelism)
  → Fast
```

### The Rule of Thumb

```
Minimum partition size: 1 GB of data per partition value

If your daily ingestion is 50 MB → do NOT partition by date
If your daily ingestion is 5 GB  → partitioning by date makes sense
```

> For GlobalMart's 1.4M orders, if the full dataset is ~200 MB, partitioning by date
> across 365 days = ~550 KB per partition. Too small. Better: no partitioning + Z-ORDER.

---
## 7. Z-ORDER BY — Clustering Without Partitioning

Z-ORDER is Delta's way of co-locating related rows within files, without creating sub-folders.

### How Z-ORDER Works

```
BEFORE OPTIMIZE ... ZORDER BY (CustomerID):
  File A: CustomerID = C001, C045, C200, C999 (random mix)
  File B: CustomerID = C002, C100, C500, C700 (random mix)

  Query: WHERE CustomerID = 'C001'
  → Must scan both File A and File B to be sure

AFTER OPTIMIZE ... ZORDER BY (CustomerID):
  File A: CustomerID = C001 – C250  (low range)
  File B: CustomerID = C251 – C500  (mid range)
  File C: CustomerID = C501 – C999  (high range)

  Query: WHERE CustomerID = 'C001'
  → Delta stats say: File A min=C001, File B min=C251 → skip B and C
  → Only File A read → much faster
```

### Running OPTIMIZE + ZORDER

```sql
-- Run after bulk loads or periodically
OPTIMIZE delta.`abfss://amazon-data@amazonprojectadls.dfs.core.windows.net/bronze/adls/orders`
ZORDER BY (CustomerID, OrderDate);
```

```python
# Or using Python:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, bronze_path)
dt.optimize().executeZOrderBy(["CustomerID", "OrderDate"])
```

### Z-ORDER vs Partitioning

| | Partitioning | Z-ORDER |
|--|-------------|----------|
| Creates sub-folders | Yes | No |
| Query pruning | Folder-level (partition pruning) | File-level (data skipping via stats) |
| Risk of small files | High (if over-partitioned) | Low |
| Cardinality limit | Low-medium cardinality only | Works with high cardinality |
| Cost | Free (at write time) | Requires OPTIMIZE run |
| Best for | Date/region/status | CustomerID, ProductID, OrderID |

---
## 8. Liquid Clustering (Delta 3.0+)

Liquid Clustering is the next evolution — it replaces both partitioning AND Z-ORDER with a unified, adaptive approach.

### The Problem With Static Partitioning

```
You partitioned by OrderChannel (3 values: Online, Mobile, In-Store)

6 months later:
  - Business starts filtering by SupplierID, not OrderChannel
  - To change partition columns → must rewrite the entire table
  - Changing partitions on a 10TB table = hours of compute + storage
```

### Liquid Clustering Solution

```sql
-- Enable when creating the table:
CREATE TABLE bronze_adls_orders
CLUSTER BY (CustomerID, OrderDate);

-- Change clustering columns any time — NO full rewrite:
ALTER TABLE bronze_adls_orders
CLUSTER BY (SupplierID, OrderDate);  ← changed, no data rewrite

-- Incrementally cluster (only uncompacted files):
OPTIMIZE bronze_adls_orders;  ← same command, no ZORDER needed
```

### Liquid Clustering vs Z-ORDER vs Partitioning

| | Partitioning | Z-ORDER | Liquid Clustering |
|--|-------------|---------|-------------------|
| Column change | Full rewrite | Easy | Easy |
| Small file risk | High | Low | Low |
| Incremental | No | No | Yes |
| Maintenance | Manual OPTIMIZE | Manual OPTIMIZE | OPTIMIZE (same cmd) |
| Availability | All Delta versions | All Delta versions | Delta 3.0+ / DBR 13.3+ |
| Best for | Simple date partitions on large tables | Medium cardinality columns | Most new tables in Databricks |

> **Recommendation for new tables on Databricks:** Use Liquid Clustering. For older tables or simple date-based splits on very large tables, partitioning still works well.

---

## 9. Decision Framework

```
What is my daily data volume per partition?
      |
      ├── < 1 GB per day
      │       → DO NOT partition
      │       → Use Z-ORDER or Liquid Clustering on query columns
      │
      └── > 1 GB per day
              |
              ├── Query pattern is date-based, low cardinality
              │       → Partition by ingestion_date or date column
              │       → Add Z-ORDER for secondary filter columns
              │
              └── Query pattern is multi-dimensional, may change
                      → Use Liquid Clustering (if DBR 13.3+)
```

---

## Key Takeaways

1. **Always use Delta** for Bronze/Silver/Gold — not CSV, not Parquet alone
2. **Partitioning = folder-level pruning** — great for date columns on large tables
3. **Over-partitioning = small file problem** — need > 1 GB per partition to justify it
4. **Z-ORDER = file-level skipping** — for high-cardinality columns without folder explosion
5. **Liquid Clustering = future default** — adaptive, incremental, no rewrite on column change
6. **For Bronze:** `ingestion_date` partition if volume justifies, otherwise Z-ORDER

---

## Discussion Questions

1. *Bronze table has 200 MB of data. A student suggests partitioning by `ingestion_date`. What's the problem?*

2. *You have 100 million rows partitioned by `CustomerID`. How many partitions could you have? Why is this dangerous?*

3. *A query filters by both `OrderDate` and `CustomerID`. You've Z-ORDERed by `OrderDate`. Does Delta skip files for the `CustomerID` filter too?*

4. *What is the main advantage of Liquid Clustering over static partitioning when query patterns change?*

5. *How does the Delta transaction log enable data skipping without partitioning?*